## **📊 Dataset Overview**

আমাদের এই প্রজেক্টের ডেটাসেটটি মূলত AI Chatbot ব্যবহার করে সিন্থেটিকভাবে (Synthetically) তৈরি করা হয়েছে।

* **Total Records:** ১৭২টি সাধারণ জ্ঞান (GK) প্রশ্ন এবং তার উত্তরের জোড়া (Question-Answer pairs)।
* **Structure:** ডেটাসেটে প্রধানত দুটি কলাম রয়েছে— একটিতে Question এবং অন্যটিতে Answer।

### 🧩 Problem Statement & Challenges

মডেলের সাথে ইউজার যেন রিয়েল-লাইফ চ্যাট করছে, এমন একটা ফিল আনার জন্য ডেটাসেটে কিছু ভ্যারিয়েশন বা চ্যালেঞ্জ রাখা হয়েছে:

* **Conversational Noise:** প্রশ্নের সাথে অনেক সময় অতিরিক্ত কথা বা চ্যাটিং টোন যুক্ত করা আছে।
* *Example 1:* What is the Capital of "France"? $\rightarrow$ Paris
* *Example 2:* Hey, What about this: "What instrument measure pressure." $\rightarrow$ barometer


* **Punctuation Noise:** ডেটাসেটে সিঙ্গেল কোটেশন ('), ডাবল কোটেশন (") এবং অন্যান্য পাংচুয়েশন মার্ক আছে, যা মডেলকে কনফিউজ করতে পারে।

### 🎯 Project Scope & Preprocessing Strategy

ফ্রম স্ক্র্যাচ (From scratch) মডেল তৈরি করার সময় আমাদের মডেলের ক্যাপাসিটি এবং ডেটার গঠন নিয়ে ভাবতে হয়।

* **Target Output (Single Word Prediction):** আমাদের ডেটাসেটে কিছু উত্তর সিঙ্গেল ওয়ার্ডের (যেমন: Paris) আবার কিছু ডাবল বা তার বেশি ওয়ার্ডের (যেমন: Washington DC)। তবে যেহেতু আমরা বেসিক RNN দিয়ে স্ক্র্যাচ থেকে মডেল বানাচ্ছি, তাই মডেলের আর্কিটেকচার সিম্পল রাখার স্বার্থে আমরা আপাতত **Single Word Prediction**-এর দিকে ফোকাস করব।
* **Data Cleaning:** মডেল ট্রেনিংয়ের আগে আমাদের একটি প্রি-প্রসেসিং (Preprocessing) পাইপলাইন বানাতে হবে। টেক্সট থেকে সব ধরনের সিঙ্গেল কোট, ডাবল কোট এবং অপ্রয়োজনীয় পাংচুয়েশন সরিয়ে ডেটা একদম ক্লিন করতে হবে, যাতে মডেল শুধু আসল শব্দগুলোর ওপর ফোকাস করতে পারে।

# **🚀 Overall Flow of the Project (GK Q&A System using RNN)**

### 📂 1. Dataset Section

* Loads the Question–Answer data synthetically generated for the GK model.

### 🧹 2. Text Preprocessing (Tokenization)

In [ ]:
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()

* **✅ What it does:** Cleans the text by removing punctuations (like `?` and `'`), converts everything to lowercase, and splits the sentence into a list of words.
* **Example:** `"What is France?"` $\rightarrow$ `["what", "is", "france"]`
* **🎯 Why necessary:** Neural networks cannot understand raw text strings. We must convert them into a structured format (tokens) before passing them to the vocabulary.

### 📖 3. Vocabulary Building

In [ ]:
vocab = {'<UNK>':0}
def build_vocab(row):
    ...

* **✅ What it does:** Assigns each unique word in the dataset a specific integer/number.
* **Example:** `"what"` $\rightarrow$ `1`, `"is"` $\rightarrow$ `2`, `"france"` $\rightarrow$ `53`
* **🎯 Why necessary:** Models only understand numerical matrices, not English words. The vocabulary acts as a bridge between human text and machine numbers.

### 🔢 4. Text $\rightarrow$ Numerical Conversion

In [ ]:
def text_to_indices(text, vocab):
    ...

* **✅ What it does:** Converts the tokenized words into their corresponding integer values from the vocabulary.
* **Example:** `"What is France"` $\rightarrow$ `[1, 2, 53]`
* **🎯 Why necessary:** This is the required input format for PyTorch Embedding layers. It also gracefully handles unknown words using the `<UNK>` token (index 0).

### 📦 5. Custom Dataset Class

In [ ]:
class QADataset(Dataset):
    ...

* **✅ What it does:** Wraps our processed data into a standard PyTorch format and returns both the **question tensor** and the **answer tensor** during iteration.
* **🎯 Why necessary:** PyTorch training strongly relies on Dataset objects to enable automatic batching, shuffling, and efficient memory management.

### 🚚 6. DataLoader

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

* **✅ What it does:** Feeds the formatted dataset to the model in manageable batches (here, one by one).
* **🎯 Why necessary:** Ensures efficient training and randomization (shuffling), which prevents the model from memorizing the order of questions and improves real learning.

### 🏗️ 7. Model Architecture (Simple RNN)

In [ ]:
class SimpleRNN(nn.Module):
    ...

* **(a) Embedding Layer:** `self.embedding = nn.Embedding(vocab_size, 50)`
* **✅ What it does:** Converts a word index into a 50-dimensional dense vector.
* **🎯 Why necessary:** Transforms isolated numbers into meaningful vector representations where words with similar contexts get similar values.


* **(b) RNN Layer:** `self.rnn = nn.RNN(50, 64, batch_first=True)`
* **✅ What it does:** Processes the sequence of word vectors step-by-step and maintains a hidden state (memory) of size 64.
* **🎯 Why necessary:** Captures the sequence information and understands the order/context of the words in the question.


* **(c) Fully Connected Layer:** `self.fc = nn.Linear(64, vocab_size)`
* **✅ What it does:** Converts the final hidden state into an output prediction matching the vocabulary size.
* **🎯 Why necessary:** Maps the RNN's internal memory output back to the vocabulary space to predict the most logical answer word.



### ➡️ 8. Forward Pass

In [ ]:
def forward(self, question):
    ...

* **Flow:** `Question` $\rightarrow$ `Embedding` $\rightarrow$ `RNN` $\rightarrow$ `Final Hidden State` $\rightarrow$ `FC Layer` $\rightarrow$ `Output Score`
* **🎯 Why necessary:** Explicitly defines the mathematical route data takes to flow through the network to generate a prediction.

### ⚙️ 9. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(...)

* **✅ What it does:** `criterion` acts as the Loss Function to measure the prediction error. `optimizer` (Adam) is the algorithm that updates the weights based on that error.
* **🎯 Why necessary:** Without loss, there is no signal to tell the model it made a mistake. Without an optimizer, the model's weights remain random and no learning/improvement occurs.

### 🔄 10. Training Loop

In [ ]:
for epoch in range(epochs):
    ...

* **Flow:**
1. Forward pass (Make prediction)
2. Compute loss (Calculate error)
3. Clear Gradients (`optimizer.zero_grad()`)
4. Backpropagation (`loss.backward()` / BPTT)
5. Update weights (`optimizer.step()`)



### 🔮 11. Prediction Function (Inference)

In [ ]:
def predict(model, question):
    ...

* **✅ What it does:** Takes a raw string question from the user, converts it to indices, runs it through the trained model, and converts the numerical output back into a human-readable word.
* **🎯 Why necessary:** This is the ultimate goal! It allows real-world, interactive usage of the system after the training phase is complete.

### 📊 12. Softmax + Argmax

In [ ]:
probs = torch.softmax(output, dim=1)
value, index = torch.max(probs, dim=1)

* **✅ What it does:** Softmax converts the raw output scores (logits) into a probability distribution (0 to 1). Argmax picks the index with the highest probability.
* **🎯 Why necessary:** The model natively outputs raw positive and negative scores. We need to convert these into clear probabilities to interpret which word from the vocabulary is the absolute best answer.

# **Import Dataset**

In [35]:
import pandas as pd

# Modified direct download link
url = "https://drive.google.com/uc?export=download&id=1lZrl1CZJtDAlbNR3CTaYlo8WWr-AvoOA"
df = pd.read_csv(url)

df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [36]:
df["question"]

,question
0,"What is the capital of ""France""?"
1,"What is the capital of ""Germany""?"
2,"What is the capital of ""Italy""?"
3,"What is the capital of ""Spain""?"
4,"What is the capital of ""Japan""?"
...,...
166,"Can you tell me, ""Which bird cannot fly""?"
167,"I was wondering, ""What is 'H2O' commonly called""?"
168,"I was wondering, ""Which planet is known as the..."
169,"Can you tell me, ""What is the main religion in..."


In [37]:
df["question"][0]

'What is the capital of "France"?'

# **Tokenize Text**

### ⚙️ কোডের ধাপে ধাপে ব্যাখ্যা (Step-by-Step Explanation)

**১. `text.lower()` (লোয়ারকেস করা):**
কম্পিউটারের কাছে "Apple" এবং "apple" দুটি সম্পূর্ণ ভিন্ন শব্দ। মডেল যেন এই কনফিউশনে না পড়ে, তাই পুরো বাক্যকে ছোট হাতের অক্ষরে (lowercase) রূপান্তর করা হয়।

**২. `re.sub(r"[\"']", "", text)` (কোটেশন মুছে ফেলা):**
আমাদের ডেটাসেটে যে ডাবল কোটেশন (`"`) এবং সিঙ্গেল কোটেশন (`'`) আছে, এই লাইনটি সেগুলোকে খুঁজে বের করে এবং একদম মুছে ফেলে (Replace with empty string `""`)।

**৩. `re.sub(r"[^a-z0-9\s]", " ", text)` (পাংচুয়েশন সরানো):**
কোটেশন বাদেও বাক্যে কমা, ফুলস্টপ, প্রশ্নবোধক চিহ্ন ইত্যাদি থাকতে পারে। এই লাইনটি ইংরেজি লেটার, নাম্বার এবং স্পেস বাদে বাকি সব ধরনের স্পেশাল ক্যারেক্টার বা পাংচুয়েশনকে খুঁজে বের করে এবং সেগুলোর জায়গায় একটি স্পেস (`" "`) বসিয়ে দেয়।

**৪. `re.sub(r"\s", " ", text).strip()` (অতিরিক্ত স্পেস ঠিক করা):**
অনেক সময় বাক্যের মাঝে ট্যাব (Tab) বা নতুন লাইন (Newline) থাকতে পারে। এটি সেগুলোকে সাধারণ স্পেসে রূপান্তর করে। আর `.strip()` ফাংশনটি বাক্যের একদম শুরুতে এবং শেষে থাকা অবাঞ্ছিত স্পেসগুলোকে কেটে বাদ দিয়ে দেয়।

**৫. `text.split()` (টোকেনাইজ করা):**
সবশেষে, এই পরিষ্কার করা পুরো বাক্যটিকে ভেঙে শব্দের একটি লিস্ট (List) বা অ্যারে তৈরি করে। মজার ব্যাপার হলো, বাক্যের মাঝে যদি একাধিক স্পেসও থাকে, `split()` সেগুলোকে ইগনোর করে শুধু আসল শব্দগুলোকে আলাদা করে নেয়।

* **উদাহরণ:** `"what is france"` হয়ে যায় `['what', 'is', 'france']`।

---

### 🔬 Regular Expression (Regex) কীভাবে কাজ করছে?

Regular Expression বা Regex হলো টেক্সটের ভেতর নির্দিষ্ট কোনো প্যাটার্ন (Pattern) খোঁজার একটি শক্তিশালী হাতিয়ার। এখানে `re.sub(pattern, replacement, text)` ফাংশনটি ব্যবহার করা হয়েছে, যার মানে হলো— "নির্দিষ্ট **pattern** খুঁজে বের করো এবং তার জায়গায় **replacement** বসিয়ে দাও।"

এখানে যে ৩টি প্যাটার্ন ব্যবহার করা হয়েছে, তার মেকানিজম নিচে দেওয়া হলো:

#### 1. `r"[\"']"`

* `r`: এর মানে হলো Raw String। এটি পাইথনকে বলে যে ভেতরের ব্যাকস্ল্যাশগুলোকে (`\`) যেন সাধারণ টেক্সট হিসেবে ধরা হয়।
* `[]`: ব্র্যাকেটের ভেতরের যেকোনো একটি ক্যারেক্টারকে সে খুঁজবে।
* `\"`: এটি হলো ডাবল কোটেশন (যেহেতু স্ট্রিংটি নিজে ডাবল কোট দিয়ে শুরু, তাই একে এস্কেপ করার জন্য `\` দেওয়া হয়েছে)।
* `'`: এটি হলো সিঙ্গেল কোটেশন।
* **সামারি:** "পুরো টেক্সটের যেখানেই ডাবল কোটেশন বা সিঙ্গেল কোটেশন পাবে, তাকে ধরবে।"

#### 2. `r"[^a-z0-9\s]"` (সবচেয়ে গুরুত্বপূর্ণ প্যাটার্ন)

* `[]`: ক্যারেক্টার সেট।
* `^`: স্কয়ার ব্র্যাকেটের ঠিক ভেতরে শুরুতে `^` (ক্যারেট) থাকার মানে হলো **NOT** (না-বোধক)। অর্থাৎ, ভেতরে যা লেখা আছে, সেগুলো *বাদে* বাকি সব!
* `a-z`: a থেকে z পর্যন্ত সব ছোট হাতের অক্ষর।
* `0-9`: 0 থেকে 9 পর্যন্ত সব সংখ্যা।
* `\s`: যেকোনো ধরনের হোয়াইটস্পেস (যেমন: স্পেস, ট্যাব)।
* **সামারি:** "ছোট হাতের অক্ষর, সংখ্যা এবং স্পেস **বাদে** অন্য যা কিছু পাবে (যেমন: ?, !, @, #, ,), সেগুলোকে ধরে একটি স্পেসে রূপান্তর করবে।"

#### 3. `r"\s"`

* `\s`: এটি রেজেক্সের একটি স্পেশাল ক্যারেক্টার ক্লাস, যা যেকোনো ধরনের ফাঁকা জায়গাকে (Space, Tab `\t`, Newline `\n`) নির্দেশ করে।
* **সামারি:** "যেকোনো ধরনের ফাঁকা জায়গাকে ধরে একটি নরমাল সিঙ্গেল স্পেসে রূপান্তর করবে।"

In [38]:
import re
def tokenize(text):
  # 1 Lowecase
  text = text.lower()

  # 2 remove quotes
  text = re.sub(r"[\"']", "", text)

  # 3 remove all punctuations
  text = re.sub(r"[^a-z0-9\s]", " ", text)

  # 4 remove extra space
  text = re.sub(r"\s", " ", text).strip()

  # tokenize split into words
  text = text.split()

  return text

In [39]:
print(tokenize(df["question"][0]))

['what', 'is', 'the', 'capital', 'of', 'france']


# **Form Vocabulary**

মডেলকে শব্দ চেনানোর জন্য আমাদের একটি ডিকশনারি বা ভোকাভুলারি তৈরি করতে হবে। এই ধাপে আমরা প্রতিটি ইউনিক শব্দকে একটি নির্দিষ্ট নাম্বার (Index) দেব।

**১. `<UNK>` টোকেন দিয়ে শুরু (The Starting Point)**
আমরা একদম ফাঁকা `{}` ডিকশনারি দিয়ে কাজ শুরু করব না। এর বদলে আমরা একটি Unknown ভেরিয়েবল দিয়ে শুরু করব:
`vocab = {'<UNK>': 0}`

* **কারণ:** মডেল ট্রেনিংয়ের পর যখন নতুন কোনো শব্দ ফেস করবে (যেমন: "What is ABC", যেখানে "ABC" আমাদের ডেটাসেটে নেই), তখন মডেল যেন কনফিউজড হয়ে এরর না দেয়। সে এই অচেনা শব্দগুলোকে `<UNK>` বা `0` ইনডেক্স হিসেবে ধরে নেবে।
* এই টোকেনটি যুক্ত করার ফলে শুরুতেই আমাদের ডিকশনারির সাইজ বা Length হয়ে যায় `1`।

**২. Question এবং Answer একত্রীকরণ**
ভোকাভুলারি তৈরির সময় আমরা Question এবং Answer-এর শব্দগুলোকে আলাদা না রেখে একসাথে মিলিয়ে ফেলব, যাতে মডেল সব শব্দ চিনতে পারে।

* **উদাহরণ:**
* Q: "What is the capital of France"
* A: "Paris"
* একত্রে প্রসেস হবে: `what is the capital of france paris`



**৩. ইনডেক্সিং ম্যাজিক (`vocab[token] = len(vocab)`)**
নতুন কোনো শব্দ ডিকশনারিতে যুক্ত করার সময় তার ইনডেক্স বা ভ্যালু হবে ডিকশনারির বর্তমান Length।

* শুরুতে ডিকশনারি: `{'<UNK>': 0}` (বর্তমান Length = 1)
* "what" যুক্ত হলে ভ্যালু হবে 1 $\rightarrow$ `{'<UNK>': 0, 'what': 1}` (Length = 2)
* "is" যুক্ত হলে ভ্যালু হবে 2 $\rightarrow$ `{'<UNK>': 0, 'what': 1, 'is': 2}` (Length = 3)
* এভাবেই চলতে থাকবে: "capital" = 3, "of" = 4, "france" = 5 ইত্যাদি।

**৪. ইউনিক শব্দ ফিল্টারিং (Unique Words Only)**
ডিকশনারিতে যেন একই শব্দ বারবার যুক্ত হয়ে জায়গা নষ্ট না করে, তাই আমরা চেক করব শব্দটি ডিকশনারিতে আগে থেকেই আছে কি না (`if word not in vocab`)।

* **উদাহরণ:** এরপর যদি নতুন প্রশ্ন আসে— "What is the capital of Germany"
* লুপ চলার সময় "what", "is", "the", "capital", "of" শব্দগুলো ডিকশনারিতে আগে থেকেই থাকায় এগুলো ইগনোর হবে।
* শুধু নতুন শব্দ হিসেবে **"Germany"** ডিকশনারিতে ঢুকবে এবং তার ইনডেক্স হবে `6`।


In [40]:
vocab = {'<UNK>' : 0}

In [41]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])


  merged_tokens = tokenized_question + tokenized_answer


  for token in merged_tokens:
      if token not in vocab:
          vocab[token] = len(vocab)

In [42]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [43]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 'iraq

In [44]:
len(vocab)

243

### 💻 Code Explanation: Building the Vocabulary

এই কোড ব্লকটির মাধ্যমে আমরা আমাদের ডেটাসেটের (DataFrame) প্রতিটি সারি (row) থেকে প্রশ্ন এবং উত্তরগুলোকে টোকেনাইজ করে একটি গ্লোবাল ডিকশনারি বা Vocabulary তৈরি করছি।

**১. Initializing the Vocabulary:**

```python
vocab = {'<UNK>' : 0}

```

* এখানে আমরা একটি গ্লোবাল ডিকশনারি `vocab` তৈরি করেছি। শুরুতেই `<UNK>` (Unknown) টোকেনটিকে `0` ইনডেক্সে রাখা হয়েছে। পরবর্তীতে মডেল যদি এমন কোনো শব্দ দেখে যা সে আগে কখনো দেখেনি, তখন সে এটিকে `0` হিসেবে ধরে নেবে।

**২. The `build_vocab` Function:**

```python
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

```

* এটি একটি কাস্টম ফাংশন, যা ডেটাসেটের প্রতিটি রো (row) বা সারি নিয়ে কাজ করে।
* এটি সারির `question` এবং `answer` কলামের বাক্যগুলোকে আগের তৈরি করা `tokenize()` ফাংশনের মাধ্যমে ছোট ছোট শব্দের লিস্টে (List) পরিণত করে।

**৩. Merging and Indexing:**

```python
    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

```

* `merged_tokens`: প্রশ্ন এবং উত্তরের দুটি আলাদা লিস্টকে যোগ করে একটি সিঙ্গেল লিস্ট বানানো হয়েছে, যাতে লুপ চালাতে সুবিধা হয়।
* `for token in merged_tokens`: এরপর লিস্টের প্রতিটি শব্দ বা টোকেন ধরে ধরে লুপ চালানো হয়েছে।
* `if token not in vocab`: চেক করা হচ্ছে শব্দটি ডিকশনারিতে আগে থেকেই আছে কি না। যদি না থাকে, তবে ডিকশনারির বর্তমান সাইজ বা `len(vocab)`-কেই ওই নতুন শব্দের ইনডেক্স ভ্যালু হিসেবে সেট করে দেওয়া হচ্ছে।

**৪. Applying the Function to the Dataset:**

```python
df.apply(build_vocab, axis=1)

```

* **`df.apply`**: এটি `pandas`-এর একটি অত্যন্ত শক্তিশালী ফাংশন। সাধারণত ডেটাসেটে `for` লুপ চালানো অনেক ধীরগতির হয়, তাই `apply` ব্যবহার করা হয়।
* **`axis=1`**: এর মানে হলো ফাংশনটি কলাম ধরে নয়, বরং **সারি ধরে (row-wise)** কাজ করবে। অর্থাৎ, এটি ডেটাসেটের প্রথম সারির প্রশ্ন-উত্তর নেবে, ডিকশনারি আপডেট করবে; তারপর দ্বিতীয় সারিতে যাবে, এভাবে পুরো ডেটাসেট স্ক্যান করবে।

**৫. Checking the Output:**

```python
vocab
len(vocab)

```

* লুপ শেষে `vocab` কল করলে পুরো ডিকশনারিটি তার শব্দ এবং ইনডেক্স নাম্বারসহ প্রিন্ট হবে।
* `len(vocab)` কল করলে আমরা দেখতে পাব আমাদের ডেটাসেটে মোট কতগুলো ইউনিক শব্দ (Unknown টোকেনসহ) আছে। এটিই হলো আমাদের মডেলের **Vocabulary Size**, যা পরবর্তীতে মডেল আর্কিটেকচার (Embedding Layer) তৈরির সময় কাজে লাগবে।

# **Text to Indices**

In [45]:
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [46]:
text_to_indices('"What is ABC', vocab)

[1, 2, 0]

### 🔢 Text to Numerical Conversion (Text to Indices)

মডেল যেহেতু সরাসরি ইংরেজি শব্দ পড়তে পারে না, তাই আমাদের ডেটাসেটের প্রতিটি বাক্যকে নাম্বারের লিস্টে বা ইনডেক্সে রূপান্তর করতে হবে। এই কোড ব্লকটি ঠিক সেই কাজটিই করছে।

**💻 The Code:**

```python
def text_to_indices(text, vocab):
    # ইনডেক্সগুলো সেভ করার জন্য একটি ফাঁকা লিস্ট
    indexed_text = []

    # বাক্যটিকে ক্লিন করে শব্দে ভাঙা হচ্ছে এবং লুপ চালানো হচ্ছে
    for token in tokenize(text):
        # শব্দটি ডিকশনারিতে থাকলে তার নাম্বার লিস্টে যোগ হবে
        if token in vocab:
            indexed_text.append(vocab[token])
        # শব্দটি ডিকশনারিতে না থাকলে <UNK> (0) যোগ হবে
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text

```

**✅ Code Explanation (লজিক কীভাবে কাজ করছে):**

* **`tokenize(text)`:** ফাংশনে ইনপুট দেওয়া বাক্যটি (যেমন: `'"What is ABC'`) প্রথমে আমাদের আগের তৈরি করা `tokenize` ফাংশনে গিয়ে ক্লিন হবে এবং শব্দের লিস্টে (`['what', 'is', 'abc']`) পরিণত হবে।
* **`if token in vocab:`** এরপর লুপটি চেক করবে এই শব্দগুলো আমাদের গ্লোবাল ডিকশনারি বা `vocab`-এ আছে কি না।
* **`vocab[token]` ও `<UNK>`:** "what" এবং "is" যেহেতু আমাদের ডিকশনারিতে আগে থেকেই আছে, তাই লিস্টে এদের ইনডেক্স (যেমন: 1 এবং 2) যুক্ত হবে। কিন্তু "abc" শব্দটি ডিকশনারিতে নেই। তাই মডেল যেন এরর না দেয়, সেইফটি মেকানিজম হিসেবে `else` ব্লকে গিয়ে এটি `<UNK>` টোকেনের ইনডেক্স অর্থাৎ `0` যুক্ত করবে।

**🎯 Example Output Breakdown:**

```python
text_to_indices('"What is ABC', vocab)
# Output: [1, 2, 0]

```

এখানে খেয়াল করলে দেখবে:

1. `"` (কোটেশন) রিমুভ হয়ে গেছে টোকেনাইজেশনের কারণে।
2. `"What"` হয়ে গেছে `1`।
3. `"is"` হয়ে গেছে `2`।
4. `"ABC"` আমাদের ডেটাসেটে ছিল না (Unknown), তাই এটি হয়ে গেছে `0`।

এর মাধ্যমেই আমাদের ডেটাসেটের কাঁচা টেক্সটগুলো PyTorch মডেলের জন্য একদম পারফেক্ট নিউমেরিক্যাল ফরম্যাটে রেডি হয়ে গেল!

# **Dataset and Dataloader class**

In [47]:
# length of dataset
df.shape[0]

171

In [48]:
index = 0
text_to_indices(df.iloc[index]['question'], vocab)

[1, 2, 3, 4, 5, 6]

In [49]:
text_to_indices(df.iloc[index]['answer'], vocab)

[7]

In [50]:
import torch
from torch.utils.data import Dataset, DataLoader

In [51]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
      self.df = df
      self.vocab = vocab
    def __len__(self):
      return self.df.shape[0]
    def __getitem__(self, index):
      numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
      numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

      return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [52]:
dataset = QADataset(df, vocab)

In [53]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [54]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [55]:
for question, answer in dataloader:
  print(question[0], answer[0])

tensor([ 1,  2,  3,  4,  5, 33]) tensor([33, 34])
tensor([ 79,  84,   3, 161, 162]) tensor([163, 164])
tensor([  1,   2, 149, 150, 121]) tensor([151, 152])
tensor([ 1,  2,  3,  4,  5, 25]) tensor([26])
tensor([ 1,  2,  3,  4,  5, 37]) tensor([38])
tensor([234, 235, 236,   1,   2, 127, 120, 121]) tensor([128, 129, 130])
tensor([ 1,  2,  3,  4,  5, 49]) tensor([50])
tensor([240, 232, 241, 242,  93,  94,  95, 206, 207]) tensor([208])
tensor([240, 232, 241, 242,   1,   2,  89,  90,  91]) tensor([92])
tensor([230, 231,  93, 202, 203, 204]) tensor([205])
tensor([237,   1, 238, 239,  79, 114,   3, 115,   5, 116]) tensor([117, 118])
tensor([ 79,  80, 168,  95, 232, 233]) tensor([ 86, 169])
tensor([230, 231,  93, 183,   2, 101, 121, 195]) tensor([196])
tensor([  1,   2,   3, 220, 129,   5,  18]) tensor([221])
tensor([ 79,  80, 168]) tensor([ 86, 169])
tensor([230, 231,  93,  94,  95, 206, 207]) tensor([208])
tensor([79, 80, 81]) tensor([82, 83])
tensor([ 93, 183,   2, 101, 121, 195]) tensor([19

আমরা এতক্ষণ `pandas` ডেটাফ্রেম (df) আর সাধারণ ডিকশনারি নিয়ে কাজ করছিলাম। কিন্তু PyTorch-এর নিউরাল নেটওয়ার্ক সাধারণ ডেটাফ্রেম বা লিস্ট বোঝে না। সে শুধু বোঝে **Tensor** (যা মূলত PyTorch-এর নিজস্ব নাম্বারের ম্যাট্রিক্স)। এছাড়া মডেলকে একসাথে হাজার হাজার ডেটা দিলে মেমরি ক্র্যাশ করতে পারে। তাই ডেটাকে একটু একটু করে (Batch) মডেলের কাছে পাঠাতে হয়।

এই পুরো সিস্টেমটাকে ম্যানেজ করার জন্য PyTorch দুটি জিনিস ব্যবহার করে:

1. **Dataset:** এটি ডেটাকে প্রস্তুত করে। (রান্নাঘরের শেফ)
2. **DataLoader:** এটি প্রস্তুত করা ডেটাকে নির্দিষ্ট পরিমাণ অনুযায়ী মডেলের কাছে সার্ভ করে। (রেস্টুরেন্টের ওয়েটার)

এবার চলো তোমার কোডটা লাইন-বাই-লাইন ভেঙে দেখি:

---

### 📦 1. The `QADataset` Class (The Chef)

এই ক্লাসটির কাজ হলো তোমার ডেটাফ্রেম থেকে একটা একটা করে বাক্য নিয়ে তাকে নাম্বারে (Index) এবং শেষে PyTorch Tensor-এ রূপান্তর করা। এই ক্লাসের ভেতরে ৩টি নির্দিষ্ট ফাংশন থাকতে হয়:

* **`__init__(self, df, vocab)`:**
এটা হলো ক্লাসের স্টার্টআপ। যখন তুমি ক্লাসটি কল করবে, এটি তোমার কাঁচা ডেটাফ্রেম (`df`) এবং সেই ভোকাভুলারি ডিকশনারিটা (`vocab`) নিজের কাছে সেভ করে রাখবে।
* **`__len__(self)`:**
PyTorch-এর জানা দরকার তোমার ডেটাসেট কত বড়। এই ফাংশনটি শুধু তোমার ডেটাফ্রেমের মোট সারির সংখ্যা (`df.shape[0]`) রিটার্ন করে।
* **`__getitem__(self, index)`: (আসল ম্যাজিক!)**
PyTorch যখন মডেল ট্রেনিং করবে, সে এই ফাংশনটিকে একটা ইনডেক্স নাম্বার (যেমন: ০, ১ বা ৫০) দিয়ে বলবে "আমাকে এই নাম্বারের ডেটা দাও"।
* তখন এটি ওই ইনডেক্সের প্রশ্ন এবং উত্তরটাকে ধরবে।
* আমাদের আগের বানানো `text_to_indices` ফাংশন দিয়ে সেগুলোকে নাম্বারের লিস্টে বানাবে।
* শেষে `torch.tensor()` ব্যবহার করে সেই লিস্টগুলোকে PyTorch-এর টেন্সরে পরিণত করে পাঠিয়ে দেবে।



**উদাহরণস্বরূপ কোডের আউটপুট:**
`dataset[0]` কল করলে সে প্রথম লাইনটি ধরে এমন একটি আউটপুট দেয়: `(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))`। মানে, প্রশ্ন এবং উত্তরের টেক্সটগুলো এখন সুন্দর নাম্বারের টেন্সর হয়ে গেছে!

---

### 🚚 2. The `DataLoader` (The Waiter)

ডেটা তো প্রস্তুত হলো, কিন্তু মডেলকে তো খাওয়াতে হবে!

```python
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

```

* **`DataLoader`** হলো সেই ডেলিভারি বয় বা ওয়েটার।
* **`batch_size=1`:** এর মানে হলো তুমি ওয়েটারকে বলে দিচ্ছ, "মডেলকে একসাথে পুরো ডেটাসেট দেবে না, প্রতিবার শুধু ১টি করে প্রশ্ন আর উত্তরের জোড়া দেবে।" (পরে মডেল বড় হলে আমরা এটি ৩২ বা ৬৪ করে দিই)।
* **`shuffle=True`:** এটি খুবই গুরুত্বপূর্ণ। এর মানে হলো ওয়েটার সিরিয়াল অনুযায়ী (১, ২, ৩...) ডেটা দেবে না। সে র‍্যান্ডমলি ডেটা মিশিয়ে মডেলকে দেবে, যাতে মডেল উত্তরের সিরিয়াল মুখস্থ না করে আসল লজিক শিখতে পারে!

---

### 🖨️ 3. The `for` Loop Output

সবশেষে তুমি যে লুপটা চালিয়েছ:

```python
for question, answer in dataloader:
    print(question[0], answer[0])

```

এটি মূলত চেক করার জন্য যে ডেটালোডার ঠিকমতো কাজ করছে কি না।
নিচে যে অদ্ভুত নাম্বারগুলো দেখতে পাচ্ছ:

```text
tensor([ 79,  84,   3, 165]) tensor([166, 167])
tensor([234, 235, 236,  79, 170, 174, 175]) tensor([176, 177, 178])

```

এগুলো আর কিছুই নয়, তোমার সেই সাধারণ জ্ঞানের প্রশ্ন আর উত্তরগুলো! `shuffle=True` থাকায় র‍্যান্ডম বিভিন্ন প্রশ্ন থেকে শব্দগুলো ইনডেক্স নাম্বারে কনভার্ট হয়ে টেন্সর হিসেবে প্রিন্ট হচ্ছে।

**এককথায় সামারি:** কাঁচা ডেটাকে PyTorch-এর মডেলের খাওয়ার উপযোগী **Tensor**-এ কনভার্ট করে একটু একটু করে মডেলের মুখে তুলে দেওয়ার জন্যই এই `Dataset` এবং `DataLoader` ব্যবহার করা হয়!


# 🏗️ RNN Architecture Implementation Overview

## 🎯 1. High-Level Idea (Sequence $\rightarrow$ Single Output)

আমাদের এই প্রজেক্টের মডেলটি হলো একটি **Sequence to Single Output** মডেল।

* **Input:** একটি প্রশ্ন (Sequence of words বা শব্দের সারি)।
* **Output:** একটি মাত্র শব্দ (Single word answer)।

**লজিক:** যেমন ধরো, প্রশ্ন হলো "Who invented the telephone?"। এর সাধারণ উত্তর হওয়ার কথা "Alexander Graham Bell"। কিন্তু আমাদের মডেল উত্তর দেবে শুধু "Alexander" বা "Bell"। এর কারণ হলো, একাধিক শব্দের (Multiple words) উত্তর জেনারেট করতে গেলে আমাদের Encoder-Decoder আর্কিটেকচার এবং Padding-এর মতো জটিল কনসেপ্ট ব্যবহার করতে হবে। আমরা মডেলটিকে অকারণে জটিল করতে চাই না। Padding এবং অন্যান্য অ্যাডভান্সড টপিকগুলো আমরা পরবর্তীতে NLP-তে শিখব।

**The Complete Pipeline:**
Text $\rightarrow$ Numbers $\rightarrow$ Embedding $\rightarrow$ RNN $\rightarrow$ Final Hidden State $\rightarrow$ Linear $\rightarrow$ Prediction

---

## 🧠 2. Core Model Components

### (A) The Embedding Layer

`self.embedding = nn.Embedding(vocab_size, embedding_dim)`

* **Embedding কী?** সহজ কথায়, Embedding হলো শব্দের অন্তর্নিহিত অর্থ বা Semantic Meaning বোঝার একটি উপায়। এটি কাছাকাছি অর্থের শব্দের মধ্যে সম্পর্ক তৈরি করে (যেমন: King ও Queen, অথবা Running ও Shoe)।
* **`embedding_dim` কী?** এটি নির্দেশ করে যে, একটি মাত্র শব্দকে বোঝাতে আমরা কয়টি নাম্বার (Feature) ব্যবহার করব।
* **কীভাবে কাজ করে?** এটি Word Indices-কে Dense Vector-এ রূপান্তর করে।
* **Input:** `"What is France?"` $\rightarrow$ `[12, 5, 87]`
* **Output (if embedding_dim = 50):** `[[0.2, -0.1, ...],`
` [0.5, 0.3, ...],`
` [0.1, -0.7, ...]]`



### (B) RNN Layer & Parameters

`self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)`

* **`batch_first=True` কেন?** এটি PyTorch-কে বলে দেয় যে, ইনপুট শেপের প্রথম ডাইমেনশনটি হবে Batch Size। এতে মডেল ডিবাগ করা সহজ হয়।
* **Standard Shape:** `(batch_size, sequence_length, features)`


* **Sequence Length কী?** এটি হলো ইনপুট বাক্যে মোট কতগুলো টোকেন বা শব্দ আছে তার সংখ্যা।
* *উদাহরণ:* `"I love deep learning"` $\rightarrow$ Sequence Length = 4.


* **Batch Size = 1:** যেহেতু আমরা একটি করে বাক্য ইনপুট দিচ্ছি, তাই আমাদের Batch Size 1। এতে করে ইনপুটের সাইজ সমান করার জন্য কোনো Padding করার দরকার হয় না।

---

## 🔄 3. Forward Pass: Step-by-Step

মডেলের ভেতর দিয়ে ডেটা কীভাবে ফ্লো করে, তার কোড এবং লজিক নিচে দেওয়া হলো:

| Step | Code | What Happens | Output Shape |
| --- | --- | --- | --- |
| **1** | `embedded = self.embedding(question)` | Converts word indices $\rightarrow$ Dense vectors | `(1, seq_len, 50)` |
| **2** | `_, final = self.rnn(embedded)` | Processes sequence, captures context | `(1, 1, 64)` |
| **3** | `final = final.squeeze(0)` | Removes the extra dimension for final state | `(1, 64)` |
| **4** | `output = self.fc(final)` | Maps hidden state $\rightarrow$ Vocabulary scores | `(1, vocab_size)` |

### 🔍 Understanding `_, final = self.rnn(embedded)`

`nn.RNN()` সবসময় দুটি জিনিস আউটপুট হিসেবে দেয়: `(output, hidden_state)`

* **`output`:** এটি প্রতিটি টাইম-স্টেপের হিডেন স্টেট ধারণ করে। আমাদের এটি দরকার নেই বলে আমরা একে `_` (Underscore) দিয়ে ইগনোর করেছি।
* **`final`:** এটি হলো একদম শেষ টাইম-স্টেপের হিডেন স্টেট। এটি পুরো বাক্যের সামারি বা কনটেক্সট মেমরিতে ধরে রাখে, তাই আমরা শুধু এটিকেই প্রেডিকশনের জন্য ব্যবহার করি।

---

## 🔷 4. Shape Transformation & Data Flow Summary

একনজরে মডেলের এক লেয়ার থেকে আরেক লেয়ারে যাওয়ার সময় ডেটার ডাইমেনশন যেভাবে পরিবর্তন হয়:

| Layer Name | Purpose | Input Shape | Output Shape |
| --- | --- | --- | --- |
| **Input** | Raw Question as word indices | `(1, seq_len)` | `(1, seq_len)` |
| **Embedding** | Dense vector representation | `(1, seq_len)` | `(1, seq_len, 50)` |
| **RNN** | Captures sequence context step-by-step | `(1, seq_len, 50)` | `(1, 1, 64)` |
| **Final State (Squeeze)** | Summarizes full sentence, removes extra dim | `(1, 1, 64)` | `(1, 64)` |
| **Linear (FC Layer)** | Word prediction scores (Logits) | `(1, 64)` | `(1, vocab_size)` |

---

## 🔑 5. Key Concepts Review

* **Embedding:** Converts discrete words into continuous meaningful vectors.
* **RNN:** Processes sequences step-by-step and retains memory of previous words.
* **Final Hidden State:** Represents the summarized meaning of the entire sentence.
* **Linear Layer:** Maps the summarized hidden features into raw vocabulary scores.
* **Output Logits:** Unnormalized probabilities representing how likely each word is to be the correct answer.

# **How Squeeze Function Works**


টেন্সরের শেপ বা ডাইমেনশন পড়ার সবচেয়ে সহজ উপায় হলো এর ব্র্যাকেট `[` গোনা।

তোমার দেওয়া উদাহরণটি দেখি: `x = torch.tensor([[[1, 1, 3]]])`

### 🔍 Reading the Shape (ডাইমেনশন কীভাবে পড়া হয়?)

তুমি একদম ঠিক ধরেছ:

1. **প্রথম লেয়ার (সবচেয়ে বাইরের `[...` ):** এর ভেতরে কয়টি উপাদান বা লিস্ট আছে? $\rightarrow$ **১টি**।
2. **দ্বিতীয় লেয়ার (মাঝের `[...` ):** এর ভেতরে কয়টি উপাদান বা লিস্ট আছে? $\rightarrow$ **১টি**।
3. **তৃতীয় লেয়ার (সবচেয়ে ভেতরের `[...` ):** এর ভেতরে কয়টি নাম্বার আছে? $\rightarrow$ **৩টি** (1, 1, 3)।

সুতরাং, এর শেপ হলো: `torch.Size([1, 1, 3])`

---

### 🗜️ How `squeeze()` Works

`squeeze` শব্দটির আক্ষরিক অর্থ হলো "চিপে ছোট করা"। PyTorch-এ `squeeze()` ফাংশনের কাজ হলো টেন্সরের শেপ থেকে ওইসব ডাইমেনশন মুছে ফেলা, যেগুলোর মান **১ (1)**। যেগুলোর মান ১-এর চেয়ে বেশি (যেমন: ৩), সেগুলোতে এটি কোনো হাত দেয় না।

তুমি ইনডেক্স নাম্বার বলে দিয়ে নির্দিষ্ট ডাইমেনশন মুছতে পারো (Python-এর ইনডেক্স 0 থেকে শুরু হয়):

**১. Removing the 0th dimension (প্রথম `1` রিমুভ করা):**

```python
y = x.squeeze(0)
print(y.shape)
# Output: torch.Size([1, 3])
# খেয়াল করো, প্রথম ডাইমেনশনটা উধাও! ব্র্যাকেটও একটা কমে গেছে: [[1, 1, 3]]

```

**২. Removing the 1st dimension (দ্বিতীয় `1` রিমুভ করা):**

```python
z = x.squeeze(1)
print(z.shape)
# Output: torch.Size([1, 3])
# এক্ষেত্রে মাঝের ডাইমেনশনটা রিমুভ হয়েছে।

```

---

### 💡 Pro-Tip (বোনাস ট্রিক)

তুমি চাইলে কোনো ইনডেক্স নাম্বার না দিয়ে সরাসরি শুধু `.squeeze()` কল করতে পারো। এটি পুরো টেন্সর স্ক্যান করবে এবং যেখানে যেখানে `1` পাবে, সবগুলোকে একসাথে মুছে ফেলবে!

```python
x = torch.tensor([[[1, 1, 3]]]) # Shape: [1, 1, 3]

# কোনো 인ডেক্স ছাড়া কল করলে:
final_x = x.squeeze()

print(final_x.shape)
# Output: torch.Size([3])
# টেন্সরটি এখন নরমাল 1D অ্যারে হয়ে গেছে: [1, 1, 3]

```

In [56]:
x = torch.tensor([[[1, 1, 3]]])
x.shape

torch.Size([1, 1, 3])

In [57]:
y = x.squeeze(0)
y.shape

torch.Size([1, 3])

In [58]:
z = x.squeeze(1)
z.shape

torch.Size([1, 3])

In [59]:
final_x = x.squeeze()

print(final_x.shape)

torch.Size([3])


## ➕ Understanding `unsqueeze()` Function in PyTorch

`unsqueeze()` হলো ঠিক `squeeze()`-এর বিপরীত কাজ।

PyTorch-এ কাজ করার সময় অনেক ক্ষেত্রে মডেল নির্দিষ্ট শেপের ডেটা দাবি করে। যেমন, মডেল হয়তো 2D ডেটা `(Batch, Features)` আশা করছে, কিন্তু তোমার কাছে আছে 1D ডেটা `(Features)`। এই ধরনের **Shape Mismatch** বা ডাইমেনশনের গরমিল হলে PyTorch এরর দেয়।

এই সমস্যা ফিক্স করার জন্য আমরা `unsqueeze()` ব্যবহার করে টেনসরের যেকোনো জায়গায় জোর করে **১ সাইজের একটি নতুন ডাইমেনশন (1-sized dimension)** যোগ করে দিই।

### 💻 Code Explanation

```python
import torch

# একটি সাধারণ 1D টেনসর (৩টি উপাদান)
x = torch.tensor([1, 2, 3])
print(x.shape)
# Output: torch.Size([3])

# ০ নাম্বার ইনডেক্সে (শুরুতে) নতুন একটি ডাইমেনশন যোগ করা হলো
y = x.unsqueeze(0)
print(y.shape)
# Output: torch.Size([1, 3])

```

### 🧠 Mental Model (কীভাবে কাজ করে?)

ব্র্যাকেট `[` গোনার সেই আগের লজিকটি দিয়ে চিন্তা করলে বিষয়টা খুব সহজ হয়ে যায়:

* **আগে ছিল:** `[1, 2, 3]` (একটি মাত্র লেয়ার, যার ভেতর ৩টি উপাদান)।
* **`unsqueeze(0)` করার পর:** তুমি বলছ, "একদম শুরুতে (0th position) একটি নতুন ব্র্যাকেট বা লেয়ার মুড়িয়ে দাও।"
* **নতুন রূপ:** `[[1, 2, 3]]` (এখন শেপ হয়ে গেল `[1, 3]`, কারণ বাইরের লেয়ারে ১টি উপাদান এবং ভেতরের লেয়ারে ৩টি উপাদান)।

**💡 বোনাস ট্রিক:**
যদি তুমি `y = x.unsqueeze(1)` লিখতে, তাহলে নতুন ডাইমেনশনটি ১ নাম্বার পজিশনে (মাঝখানে) বসত।
তখন ডেটাটি দেখতে হতো এমন:

```python
[[1],
 [2],
 [3]]

```

এবং এর শেপ হয়ে যেত: `torch.Size([3, 1])`!


 # **RNN Architecture_s Code**

In [60]:
import torch.nn as nn

In [61]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
        super(SimpleRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, question):
      embedded = self.embedding(question)
      _, final = self.rnn(embedded)

      return self.fc(final.squeeze(0))

In [62]:
model = SimpleRNN(vocab_size = len(vocab))


# 💻 Code Explanation: Simple RNN Architecture

এই কোড ব্লকটির মাধ্যমে আমরা PyTorch ব্যবহার করে আমাদের নিউরাল নেটওয়ার্ক বা মডেলের মূল কাঠামো (Architecture) তৈরি করেছি।

পুরো ক্লাসটিকে প্রধানত দুটি অংশে ভাগ করা যায়:
১. **`__init__` (The Builder):** যেখানে মডেলের লেয়ারগুলো তৈরি করা হয়।
২. **`forward` (The Pipeline):** যেখানে ডেটা এক লেয়ার থেকে অন্য লেয়ারে যায়।

---

### 🛠️ ১. Initialization (`__init__` method)

এখানে আমরা মডেলের প্রয়োজনীয় সব যন্ত্রপাতি বা লেয়ারগুলো ডিক্লেয়ার করে রাখি।

```python
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
        super(SimpleRNN, self).__init__()

```

* **`nn.Module`:** PyTorch-এ যেকোনো মডেল বানাতে হলে তাকে এই `nn.Module` থেকে ইনহেরিট (Inherit) করতে হয়। এটি মডেলের বেস বা ফাউন্ডেশন।
* **Parameters:** আমরা ৩টি জিনিস ইনপুট নিচ্ছি— ভোকাভুলারির সাইজ, শব্দের ফিচারের সংখ্যা (`50`), এবং মেমরি বা নিউরনের সংখ্যা (`64`)।

**The Layers:**

```python
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

```

* **`self.embedding`:** এটি আমাদের সেই ডিকশনারি, যা প্রতিটি ইনডেক্স নাম্বারকে ৫০টি ফিচারের একটি ভেক্টরে রূপান্তর করবে।
* **`self.rnn`:** এটি আমাদের মেইন ব্রেইন। এটি ইনপুট হিসেবে `50` ডাইমেনশনের শব্দ নেবে এবং প্রসেস করে `64` ডাইমেনশনের মেমরি তৈরি করবে। `batch_first=True` দেওয়া হয়েছে যেন শেপের প্রথমেই ব্যাচ সাইজ থাকে।
* **`self.fc` (Fully Connected / Linear Layer):** এটি শেষ লেয়ার। এটি `64` ডাইমেনশনের মেমরি বা সামারিকে প্রসেস করে আমাদের ডিকশনারির মোট শব্দের সাইজে (`vocab_size`) রূপান্তর করে দেবে (স্কোর দেওয়ার জন্য)।

---

### 🚀 ২. The Forward Pass (`forward` method)

এই ফাংশনটি ঠিক করে দেয় যে, ইনপুট ডেটা কোন লেয়ারের পর কোন লেয়ারে যাবে এবং কীভাবে তার শেপ (Shape) পরিবর্তন হবে।

```python
    def forward(self, question):
      embedded = self.embedding(question)

```

* **Step 1:** ইনপুট `question` (যেমন: `[1, 2, 3]`) প্রথমে Embedding লেয়ারে যায়।
* *Shape Change:* `(1, seq_len) -> (1, seq_len, 50)`

```python
      _, final = self.rnn(embedded)

```

* **Step 2:** এম্বেড করা ডেটাটি RNN-এ প্রবেশ করে। `_` দিয়ে আমরা অপ্রয়োজনীয় আউটপুট ইগনোর করেছি এবং `final` ভেরিয়েবলে পুরো বাক্যের ফাইনাল সামারি বা হিডেন স্টেটটি সেভ করেছি।
* *Shape Change:* `(1, seq_len, 50) -> (1, 1, 64)`

```python
      return self.fc(final.squeeze(0))

```

* **Step 3 (Squeeze & Linear):** `fc` বা লিনিয়ার লেয়ার 3D ডেটা বোঝে না। তাই আমরা `final.squeeze(0)` ব্যবহার করে প্রথম ডাইমেনশনটি (যা লেয়ারের সংখ্যা নির্দেশ করে) মুছে ফেলেছি। এরপর সেই 2D ডেটাটি লিনিয়ার লেয়ারে পাঠিয়ে দিয়েছি চূড়ান্ত স্কোরের জন্য।
* *Shape Change:* `final.squeeze(0)` এর কারণে শেপ হয় `(1, 64)`। লিনিয়ার লেয়ার শেষে শেপ দাঁড়ায় `(1, vocab_size)`।

---

### 🏗️ ৩. Creating the Model Object

```python
model = SimpleRNN(vocab_size = len(vocab))

```

* সবশেষে, আমরা এই ক্লাসের একটি অবজেক্ট বা বাস্তব রূপ তৈরি করেছি `model` নামে।
* এখানে আমরা শুধু আমাদের ডিকশনারির সাইজটা (`len(vocab)`) পাঠিয়ে দিয়েছি। বাকি দুটি প্যারামিটার (`embedding_dim` এবং `hidden_size`) আগে থেকেই ডিফল্ট হিসেবে 50 এবং 64 সেট করা আছে, তাই আর আলাদা করে দিতে হয়নি!

---

কোড এবং থিওরির এই সুন্দর মেলবন্ধন তোমার প্রজেক্ট নোটকে এখন পুরোপুরি স্বয়ংসম্পূর্ণ করে তুলল!

# **Training Loop**

In [65]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [69]:
for epoc in range(epochs):
  total_loss = 0

  for question, answer in dataloader:
    optimizer.zero_grad()
    output = model(question)

    # fix: take only the first answer token as target -> shape(1,)
    #print(answer)
    target = answer[0][0].unsqueeze(0)

    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {total_loss:.4f}")


## 🏋️‍♂️ Code Explanation: The Training Loop

মডেল ট্রেনিংয়ের পুরো প্রক্রিয়াটিকে আমরা একটি ক্লাসরুমের সাথে তুলনা করতে পারি। যেখানে মডেল হলো ছাত্র, Loss Function হলো পরীক্ষক, এবং Optimizer হলো শিক্ষক যে ছাত্রের ভুল শুধরে দেয়।

### 🛠️ ১. Training Setup (প্রস্তুতি পর্ব)

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

```

* **`criterion` (Loss Function):** এটি হলো আমাদের পরীক্ষক। `CrossEntropyLoss` মূলত ক্লাসিফিকেশন বা সঠিক শব্দ খুঁজে বের করার প্রজেক্টে সবচেয়ে ভালো কাজ করে। এটি চেক করবে মডেলের দেওয়া উত্তর (Output) আর আসল উত্তরের (Target) মধ্যে পার্থক্য বা ভুল (Loss) কতটুকু।
* **`optimizer` (The Teacher):** `Adam` হলো বর্তমান সময়ের সবচেয়ে জনপ্রিয় অপটিমাইজার। এর কাজ হলো Loss বা ভুলের পরিমাণ দেখে মডেলের ভেতরের প্যারামিটার বা ওয়েটগুলো একটু একটু করে আপডেট করা, যাতে পরের বার ভুল কম হয়। `lr = 0.0001` (Learning Rate) মানে হলো শিক্ষক ছাত্রকে কত দ্রুত বা ধীরে শিখতে নির্দেশ দিচ্ছেন।
* **`epochs = 50`:** এর মানে হলো মডেল পুরো ডেটাসেটটি ৫০ বার পড়বে এবং পরীক্ষা দেবে।

---

### 🔄 ২. The Training Loops (ট্রেনিংয়ের মূল সাইকেল)

```python
for epoch in range(epochs):  # Outer Loop
    total_loss = 0

```

* এটি হলো **Epoch Loop**। প্রতি ইপোকের শুরুতে আমরা `total_loss = 0` করে নিচ্ছি, যেন প্রতিবার নতুন করে ভুলের হিসাব রাখা যায়।

```python
    for question, answer in dataloader:  # Inner Loop

```

* এটি হলো **Batch Loop**। আমাদের সেই ওয়েটার (`dataloader`) একেকটি প্রশ্ন ও উত্তরের জোড়া নিয়ে মডেলের কাছে আসছে।

---

### ✋ ৩. The 5 Golden Steps of PyTorch Training

এই লুপের ভেতরের ৫টি ধাপ PyTorch-এর যেকোনো মডেল ট্রেনিংয়ের জন্য একদম ফরজ (Mandatory)!

**Step 1: Zero Gradients**

```python
        optimizer.zero_grad()

```

* পাইথন আগের ব্যাচের ভুলের হিসাব (Gradients) মেমরিতে জমিয়ে রাখে। তাই নতুন প্রশ্ন দেওয়ার আগে `zero_grad()` দিয়ে আগের ভুলের হিসাব মুছে বা রিসেট করে নিতে হয়।

**Step 2: Forward Pass (উত্তর দেওয়া)**

```python
        output = model(question)

```

* মডেল প্রশ্নটি পড়ে তার নিজের মতো করে একটি উত্তর বা স্কোর (`output`) জেনারেট করল।

**Step 3: Target Setup & Calculate Loss (খাতা দেখা)**

```python
        target = answer[0][0].unsqueeze(0)
        loss = criterion(output, target)

```

* **Shape Mismatch Fix:** `CrossEntropyLoss`-এর নিয়ম হলো, একটি মাত্র উত্তরের জন্য টার্গেটের শেপ হতে হবে `(1,)` বা 1D টেনসর। কিন্তু আমাদের উত্তরের শেপ ছিল 3D বা 2D। তাই `answer[0][0]` দিয়ে মূল শব্দটি বের করে তার আগে `.unsqueeze(0)` দিয়ে একটি ডাইমেনশন যোগ করা হয়েছে, যাতে পরীক্ষক ঠিকমতো খাতা দেখতে পারে!
* এরপর `criterion` মডেলের `output` এবং আসল `target`-এর মধ্যে তুলনা করে ভুলের পরিমাণ (`loss`) বের করল।

**Step 4: Backward Pass (ভুল কোথায় বের করা)**

```python
        loss.backward()

```

* এটি মডেলের ব্যাকপ্রপাগেশন (Backpropagation)। পরীক্ষক খাতায় কোথায় কোথায় ভুল হয়েছে, তা লাল কালি দিয়ে দাগিয়ে দিল (Gradients ক্যালকুলেট করল)।

**Step 5: Optimizer Step (নিজেকে শুধরে নেওয়া)**

```python
        optimizer.step()
        total_loss += loss.item()

```

* শিক্ষক (`optimizer`) দাগানো খাতা দেখে মডেলের ভেতরের মেমরি বা নিউরনের ভ্যালুগুলো একটু পরিবর্তন করে দিল, যাতে এরপর এই ভুল আর না হয়।
* শেষে বর্তমান ব্যাচের ভুলটা `total_loss`-এ যোগ করে রাখা হলো।

---

### 🖨️ ৪. Tracking Progress (অগ্রগতি দেখা)

```python
    # এই ব্লকটি dataloader লুপের বাইরে (অর্থাৎ epoch লুপের সাথে সমান) রাখতে হবে
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} Loss: {total_loss:.4f}")

```

* পুরো ডেটাসেট পড়া শেষ হলে (১ ইপোক শেষে), আমরা চেক করছি ইপোক নাম্বারটা ১০ দিয়ে ভাগ করা যায় কি না (`% 10 == 0`)।
* এর মানে হলো, আমরা প্রতি ইপোকে প্রিন্ট না করে, প্রতি ১০ ইপোক পরপর (যেমন: ১০, ২০, ৩০...) টোটাল লসের পরিমাণটা স্ক্রিনে প্রিন্ট করে দেখব যে মডেল ঠিকমতো শিখছে কি না। (লসের পরিমাণ আস্তে আস্তে কমতে থাকলে বুঝতে হবে মডেল দারুণ শিখছে!)

---

**💡 ছোট্ট টিপস:** তোমার অরিজিনাল কোডে `epoc` বানানটা একটু টাইপো ছিল এবং `print` ব্লকটা ভেতরের `dataloader` লুপের ভেতর ঢুকে গিয়েছিল।

# **Prediction**

In [70]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9), ('italy', 10), ('rome', 11), ('spain', 12), ('madrid', 13), ('japan', 14), ('tokyo', 15), ('canada', 16), ('ottawa', 17), ('brazil', 18), ('brasilia', 19), ('australia', 20), ('canberra', 21), ('india', 22), ('new', 23), ('delhi', 24), ('china', 25), ('beijing', 26), ('russia', 27), ('moscow', 28), ('united', 29), ('states', 30), ('washington', 31), ('dc', 32), ('mexico', 33), ('city', 34), ('egypt', 35), ('cairo', 36), ('turkey', 37), ('ankara', 38), ('argentina', 39), ('buenos', 40), ('aires', 41), ('south', 42), ('korea', 43), ('seoul', 44), ('indonesia', 45), ('jakarta', 46), ('pakistan', 47), ('islamabad', 48), ('bangladesh', 49), ('dhaka', 50), ('nepal', 51), ('kathmandu', 52), ('sri', 53), ('lanka', 54), ('colombo', 55), ('thailand', 56), ('bangkok', 57), ('malaysia', 58), ('kuala', 59), ('lumpur', 60), ('vietnam', 61), ('hanoi', 62

In [71]:
idx_to_word = {idx : word for word, idx in vocab.items()}

In [72]:
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france',
 7: 'paris',
 8: 'germany',
 9: 'berlin',
 10: 'italy',
 11: 'rome',
 12: 'spain',
 13: 'madrid',
 14: 'japan',
 15: 'tokyo',
 16: 'canada',
 17: 'ottawa',
 18: 'brazil',
 19: 'brasilia',
 20: 'australia',
 21: 'canberra',
 22: 'india',
 23: 'new',
 24: 'delhi',
 25: 'china',
 26: 'beijing',
 27: 'russia',
 28: 'moscow',
 29: 'united',
 30: 'states',
 31: 'washington',
 32: 'dc',
 33: 'mexico',
 34: 'city',
 35: 'egypt',
 36: 'cairo',
 37: 'turkey',
 38: 'ankara',
 39: 'argentina',
 40: 'buenos',
 41: 'aires',
 42: 'south',
 43: 'korea',
 44: 'seoul',
 45: 'indonesia',
 46: 'jakarta',
 47: 'pakistan',
 48: 'islamabad',
 49: 'bangladesh',
 50: 'dhaka',
 51: 'nepal',
 52: 'kathmandu',
 53: 'sri',
 54: 'lanka',
 55: 'colombo',
 56: 'thailand',
 57: 'bangkok',
 58: 'malaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'vietnam',
 62: 'hanoi',
 63: 'uae',
 64: 'abu',
 65: 'dhabi',
 66: 'iran',
 67: 'tehran',
 68: '

In [73]:
def predict(question_text, model, vocab, idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text, vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0) # batch size, sequence length

    pred = torch.argmax(model(x), dim=1).item()

  return idx_to_word.get(pred, "<UNK>")


In [78]:
# Test
tests = [
    "Hey, do you know What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:50} → {predict(q, model, vocab, idx_to_word)}")

Hey, do you know What is the capital of France?    → paris
What is H2O commonly called?                       → water
Which planet is known as the red planet?           → mars
Which bird cannot fly?                             → ostrich
What is the capital of Japan ?                     → tokyo


## 🎯 Code Explanation: The Prediction Phase (Inference)

এই ধাপে আমরা আমাদের ট্রেইন করা মডেলকে নতুন কিছু প্রশ্ন করে দেখব সে সঠিক উত্তর দিতে পারে কি না। এই পুরো প্রক্রিয়াটিকে মেশিন লার্নিংয়ের ভাষায় **Inference** বলা হয়।

### 🔄 ১. Reverse Dictionary (নাম্বার থেকে শব্দে ফেরা)

```python
idx_to_word = {idx : word for word, idx in vocab.items()}

```

* **কেন লাগবে?** আমাদের মডেল তো আর ইংরেজি শব্দ বোঝে না, সে আউটপুট হিসেবে দেবে একটি ইনডেক্স নাম্বার (যেমন: `45`)। কিন্তু মানুষের তো নাম্বার দেখে বোঝার উপায় নেই ৪৫ মানে কী!
* তাই আমরা আগের `vocab` (যেখানে ছিল `word: index`) ডিকশনারিটাকে উল্টে দিয়ে `idx_to_word` (অর্থাৎ `index: word`) বানালাম। যাতে মডেল নাম্বার দিলে আমরা সহজে ওই নাম্বারের পেছনের আসল শব্দটা খুঁজে বের করতে পারি।

---

### 🧠 ২. The Prediction Function (মডেলের পরীক্ষা)

```python
def predict(question_text, model, vocab, idx_to_word):
  model.eval()

```

* **`model.eval()`:** এটি PyTorch-এর একটি অত্যন্ত গুরুত্বপূর্ণ রুল! এর মানে হলো মডেলকে বলে দেওয়া, "তোমার শেখার সময় (Training) শেষ, এখন পরীক্ষা দেওয়ার সময়!" এটি মডেলের ভেতরের কিছু রেন্ডমনেস (যেমন Dropout বা BatchNorm থাকলে) বন্ধ করে দেয়, যাতে সে ফোকাসড হয়ে শুধু উত্তর দিতে পারে।

```python
  with torch.no_grad():

```

* **`torch.no_grad()`:** ট্রেনিংয়ের সময় মডেল ভুল থেকে শেখার জন্য Gradients (ভুলের হিসাব) মেমরিতে সেভ করত। কিন্তু এখন তো সে আর শিখবে না, শুধু উত্তর দেবে। তাই এই কমান্ড দিয়ে আমরা ব্যাকগ্রাউন্ডের সব মেমরি ক্যালকুলেশন বন্ধ করে দিলাম। এতে কোড অনেক ফাস্ট রান করে এবং মেমরি বেঁচে যায়।

```python
    indices = text_to_indices(question_text, vocab)
    
    if not indices:
      return "<UNK>"

```

* **Data Prep:** নতুন প্রশ্নটিকে আগের মতোই টোকেনাইজ করে নাম্বারে (`indices`) রূপান্তর করা হলো। যদি এমন কোনো অদ্ভুত প্রশ্ন আসে যার কোনো শব্দই ডিকশনারিতে নেই, তবে সরাসরি `<UNK>` রিটার্ন করে দেওয়া হবে।

```python
    x = torch.tensor(indices).unsqueeze(0) # batch size, sequence length

```

* **Shape Matching:** আমাদের মডেল তো ইনপুট হিসেবে 2D ডাইমেনশন `(batch_size, seq_len)` আশা করে। কিন্তু আমাদের `indices` হলো সাধারণ 1D লিস্ট `[1, 2, 3]`। তাই `unsqueeze(0)` দিয়ে আমরা লিস্টের শুরুতে একটি ব্র্যাকেট যোগ করে একে `[[1, 2, 3]]` বা `(1, 3)` বানিয়ে দিলাম!

```python
    pred = torch.argmax(model(x), dim=1).item()

```

* **The Magic Line:** * `model(x)`: মডেলকে প্রশ্নটি দেওয়া হলো এবং সে ডিকশনারির ২৪৩টি শব্দের জন্য ২৪৩টি আলাদা স্কোর দিল।
* `torch.argmax(...)`: ২৪৩টি স্কোরের মধ্যে যেই ইনডেক্সের স্কোর সবচেয়ে বেশি (Highest Probability), এটি সেই ইনডেক্স নাম্বারটি খুঁজে বের করে।
* `.item()`: PyTorch টেন্সরের ভেতর থেকে পিওর পাইথনের নাম্বারটি (যেমন `tensor(45)` থেকে শুধু `45`) বের করে আনে।



```python
  return idx_to_word.get(pred, "<UNK>")

```

* সবশেষে, সেই সর্বোচ্চ স্কোরের নাম্বারটিকে আমাদের উল্টো ডিকশনারি (`idx_to_word`) দিয়ে আসল ইংরেজি শব্দে রূপান্তর করে রিটার্ন করা হলো।

---

### 🧪 ৩. Real-World Testing (ফলাফল যাচাই)

```python
# Test
tests = [
    "Hey, do you know What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",
]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")

```

* এখানে আমরা একটি লিস্টে কিছু নতুন এবং ট্রিকি প্রশ্ন রেখেছি (যেমন প্রথম প্রশ্নে বাড়তি কিছু কথাবার্তা যোগ করা হয়েছে)।
* এরপর একটি সাধারণ `for` লুপ চালিয়ে প্রতিটি প্রশ্ন আমাদের `predict` ফাংশনে পাঠিয়ে প্রিন্ট করে দেখছি মডেল কতটুকু স্মার্ট হয়েছে!
